In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 12      # Input history length
PRED_LEN = 6     # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.1, 0.25, 0.5, 0.75,0.9]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1377, 18])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1377, 18])

Forecasts are now unscaled and ready for plotting.


In [2]:
all_forecasts_unscaled

{0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 41.4551, 41.7769, 41.6567],
         [34.3895, 35.0485, 35.4885,  ..., 42.7838, 43.1290, 43.0454],
         [35.0485, 35.4885, 36.1475,  ..., 43.6856, 44.0353, 43.9656],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 44.8446, 44.5081, 44.2100],
         [47.7445, 47.9640, 48.1835,  ..., 44.4083, 44.0874, 43.7988],
         [47.9640, 48.1835, 48.1835,  ..., 44.6952, 44.4286, 44.1753]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 42.8825, 43.2909, 43.4809],
         [34.3895, 35.0485, 35.4885,  ..., 44.3599, 44.8337, 45.0818],
         [35.0485, 35.4885, 36.1475,  ..., 45.2537, 45.7340, 45.9817],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 45.9655, 45.7982, 45.3486],
         [47.7445, 47.9640, 48.1835,  ..., 45.4809, 45.3279, 44.8960],
         [47.9640, 48.1835, 48.1835,  ..., 45.7302, 45.5944, 45.1995]]),
 0.5: tensor([[34.1700, 34.3895, 35.0485,  ..., 43.1748, 43.6137, 43.9109],
         [34.3895, 35.0485, 3

In [3]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [4]:
combined.shape

torch.Size([5, 1377, 18])

In [5]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 41.4551, 41.7769, 41.6567],
         [34.3895, 35.0485, 35.4885,  ..., 42.7838, 43.1290, 43.0454],
         [35.0485, 35.4885, 36.1475,  ..., 43.6856, 44.0353, 43.9656],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 44.8446, 44.5081, 44.2100],
         [47.7445, 47.9640, 48.1835,  ..., 44.4083, 44.0874, 43.7988],
         [47.9640, 48.1835, 48.1835,  ..., 44.6952, 44.4286, 44.1753]],

        [[34.1700, 34.3895, 35.0485,  ..., 42.8825, 43.2909, 43.4809],
         [34.3895, 35.0485, 35.4885,  ..., 44.3599, 44.8337, 45.0818],
         [35.0485, 35.4885, 36.1475,  ..., 45.2537, 45.7340, 45.9817],
         ...,
         [47.0850, 47.7445, 47.9640,  ..., 45.9655, 45.7982, 45.3486],
         [47.7445, 47.9640, 48.1835,  ..., 45.4809, 45.3279, 44.8960],
         [47.9640, 48.1835, 48.1835,  ..., 45.7302, 45.5944, 45.1995]],

        [[34.1700, 34.3895, 35.0485,  ..., 43.1748, 43.6137, 43.9109],
         [34.3895, 35.0485, 35.4885,  ..., 44

In [6]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 25.031620025634766
Max value in combined: 54.432796478271484
